<a href="https://colab.research.google.com/github/jinw00ch01/Dacon_RobotWorldModel_ActionVideo/blob/main/colab/train_cosmos_ac_v4aug.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# S3: Cosmos-Predict2.5-2B 행동 조건 미세조정 (Colab A100/H100)

Ultra 가 만든 잠재 캐시와 변환 가중치를 Google Drive `MyDrive/dacon_wm` 에서 읽고, 체크포인트를 같은 곳에 저장한다.
끊기면 **모두 실행**을 다시 누르면 마지막 체크포인트부터 이어서 학습한다.

준비 (사람): Drive 에 `dacon_wm` 폴더가 Ultra 에서 동기화되어 있어야 한다. 런타임 유형은 GPU (A100 또는 H100).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
SHARED = '/content/drive/MyDrive/dacon_wm'
assert os.path.isdir(SHARED), 'Drive 에 dacon_wm 폴더가 없습니다'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 코드: 공개 저장소의 main (실행한 커밋은 아래에 출력되고 체크포인트 config 에도 남는다)
!rm -rf /content/repo && git clone -q https://github.com/jinw00ch01/Dacon_RobotWorldModel_ActionVideo /content/repo
!cd /content/repo && git log --oneline -1
!pip -q install diffusers==0.40.0 peft==0.21.2 transformers==5.18.0 accelerate safetensors einops av opencv-python-headless pandas pyarrow
# Colab 에 깔린 torchao 0.10 은 peft 0.21 이 LoRA 를 만들 때 거부한다. 여기서는 쓰지 않으므로 지운다.
!pip -q uninstall -y torchao

In [ ]:
# 잠재 캐시를 로컬 디스크로 복사 (Drive 에서 직접 읽으면 느림)
!mkdir -p /content/latents && rsync -a --info=progress2 "$SHARED/latents_240x320/" /content/latents/
os.environ['WM_SHARED'] = SHARED
os.environ['WM_ACTION_STATS'] = f'{SHARED}/so100_action_statistics.json'

In [ ]:
# v4_aug: v2_long 16,000 스텝(제출 3·5 의 모델)에서 이어서, 관절 기준점 어긋남 증강을 넣어 24,000 스텝까지 학습한다.
# 증강: 클립의 절반에서 관절값을 '다른 학습 데이터셋 평균 자세 - 자기 데이터셋 평균 자세' x U(0, 1) 만큼 옮긴다 (학습 로봇들 사이의 기준점 차이 범위 그대로, 평가 데이터에서 온 숫자 없음).
#   상대·차분 특징은 그대로이고 절대 채널만 바뀌므로, 기준점이 다른 로봇에서도 영상이 흔들리지 않도록 배운다.
# 학습률은 일정(LoRA 5e-5, 임베더 1.5e-4)하고 500 스텝마다 adapter_0XXXXX.pt 를 남겨 마지막 체크포인트 평균에 쓴다.
import glob, shutil, torch
n_shards = len(glob.glob('/content/latents/*.pt'))
assert n_shards == 116, f'잠재 캐시가 {n_shards}/116 개입니다. Drive 업로드가 끝난 뒤 다시 실행하세요.'
SRC = f'{SHARED}/cosmos_ac/runs/v2_long'
RUN = f'{SHARED}/cosmos_ac/runs/v4_aug'
os.makedirs(RUN, exist_ok=True)
if not os.path.exists(f'{RUN}/adapter_last.pt'):
    for f in ('adapter_last.pt', 'optimizer_last.pt'):
        shutil.copy(f'{SRC}/{f}', f'{RUN}/{f}')
    print('v2_long 16,000 스텝에서 시작')
BATCH = 4  # v2_long 과 같은 배치
!cd /content/repo && python -m wmgen.train_cosmos_ac --latents /content/latents --out "$RUN" --batch $BATCH --steps 24000 --save-every 500 --keep-every 500 --lr 5e-5 --embedder-lr 1.5e-4 --lr-schedule constant --abs-aug-prob 0.5 --abs-aug-max-scale 1.0
